# ContextualPrecisionMetric

## What it measures

Whether the retriever put the *relevant* chunks at the top. The judge labels each node in
`retrieval_context` relevant or not with respect to `expected_output`, then computes a
weighted cumulative precision that rewards relevant nodes appearing early. A retriever
that returns the right chunk at rank 9 scores far worse than one that returns it at rank
1, even though both "found" it.

## When it is useful

When ranking, not recall, is the suspect: the corpus contains the answer and the
generator is competent, but answers are drifting because the useful chunk is being pushed
down the list by near-duplicates, headers, or boilerplate. It is also the metric that
catches an embedding-model or chunking change turning a good retriever into a mediocre one.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes |
| `retrieval_context` | yes - the ranked chunk texts, **in retrieval order** |
| `expected_output` | yes - the reference answer relevance is judged against |

`actual_output` is not part of the computation. This metric can therefore be run with no
LLM generation at all.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
# The contract these notebooks were written against. The application may
# serve a HIGHER minor version: a MINOR bump is additive by its own
# contract policy (1.0.0 -> 1.1.0 added HealthResponse.build_version and
# changed nothing else), so treating it as a mismatch would turn this
# guard into noise on every single call. Only a MAJOR change, or an
# application older than these notebooks, is a problem.
EXPECTED_SCHEMA_VERSION = "1.0.0"


def contract_version(value):
    """(major, minor) from a MAJOR.MINOR.PATCH string, or None."""
    try:
        parts = value.split("+")[0].split(".")
        return int(parts[0]), int(parts[1])
    except (AttributeError, IndexError, ValueError):
        return None

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    served_version = contract_version(served) if served else None
    expected_version = contract_version(EXPECTED_SCHEMA_VERSION)
    if served_version and served_version[0] != expected_version[0]:
        raise ApiError(
            f"The application serves contract version {served}; these notebooks were "
            f"written against {EXPECTED_SCHEMA_VERSION}. A MAJOR change means fields "
            f"may have been removed or retyped - re-derive the goldens against the "
            f"new contract rather than scoring against one they do not match."
        )
    if served_version and served_version[1] < expected_version[1]:
        print(f"WARNING: application reports contract version {served}, older than "
              f"the {EXPECTED_SCHEMA_VERSION} these notebooks were written against. "
              f"Fields the goldens rely on may not exist yet.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoint exercised

`POST /api/rag/retrieve` - retrieval only, **no LLM call**. This is the right surface for
a ranking metric: scoring retrieval through `POST /api/rag/query` would confound retriever
quality with generator behaviour, and the application documents this endpoint as existing
precisely so the two can be separated.

The response returns `chunks` already in rank order with a cosine `score`, `chunk_id`,
`document_id` and `source` per chunk, so the ordering DeepEval scores is the application's
real ordering, not something this notebook reconstructed.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
#
# `document_type: "policy"` restricts retrieval to the policy corpus, which is
# what the golden is written against. `top_k: 10` deliberately asks for more
# chunks than are relevant - a retriever that ranks the one relevant chunk
# first still scores 1.0 on precision, so padding the request makes the ranking
# quality visible instead of hiding it behind a tight top_k.
# --------------------------------------------------------------------------
QUERY = (
    "What indicators identify structuring, and what cash transaction thresholds "
    "require review under the transaction monitoring policy?"
)

request_body = {"query": QUERY, "document_type": "policy", "top_k": 10}

print("POST", f"{API_BASE}/api/rag/retrieve")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body:", json.dumps(request_body, indent=2))

In [ ]:
# --------------------------------------------------------------------------
# The raw response: rank-ordered chunks with cosine similarity scores.
# --------------------------------------------------------------------------
retrieval = api("POST", "/api/rag/retrieve", json_body=request_body)

print(f"retrieval_run_id : {retrieval['retrieval_run_id']}")
print(f"filters          : {json.dumps(retrieval['filters'])}")
print(f"latency_ms       : {retrieval['latency_ms']}")
print()
for rank, chunk in enumerate(retrieval["chunks"], start=1):
    print(f"{rank:>2}. {chunk['chunk_id']:<12} score={chunk['score']:.3f}  "
          f"section={chunk['metadata'].get('section')!r}")
    print(f"    {chunk['text'][:150].replace(chr(10), ' ')}...")

## Mapping the API response onto DeepEval fields

| DeepEval field | API field | Note |
|---|---|---|
| `input` | `query` (echoed) | The string that was actually embedded |
| `retrieval_context` | `[c["text"] for c in chunks]` | **Order preserved** - the metric's whole signal is the ranking |
| `expected_output` | derived below | A reference answer, never copied from a live response |
| `actual_output` | unused by the metric | Set to the golden so the test case is well-formed |

`retrieval_context` is the list of chunk **texts**. The `chunk_id`/`score` metadata is
printed above for debugging but is not passed to the judge, which reasons over text.

In [ ]:
# --------------------------------------------------------------------------
# Deriving the golden.
#
# Contextual Precision and Recall both need an `expected_output` - a reference
# answer. It must not be copied from a live response, or the metric would be
# grading the application against itself.
#
# The authoritative source is the policy document the application indexes. It
# is reachable black-box at GET /api/documents/{document_id}, which returns the
# full markdown `content`. The reference answer below is a faithful restatement
# of AML-001 sections 2.1, 2.2 and 2.3. The assertion that follows proves those
# clauses really are in the served document, so the golden cannot silently
# drift away from the corpus.
# --------------------------------------------------------------------------
policy_docs = [d for d in api("GET", "/api/documents", params={"type": "policy"})]
print("policy documents served by the API:")
for d in policy_docs:
    print(f"  document_id={d['document_id']}  {d.get('title')}  ({d.get('source')})")

AML001 = next(d for d in policy_docs if "AML-001" in (d.get("source") or ""))
aml001_content = api("GET", f"/api/documents/{AML001['document_id']}")["content"]

EXPECTED_OUTPUT = (
    "Under AML-001, any single cash transaction of GBP 10,000 or more must be recorded "
    "and reviewed before processing, and multiple cash transactions by or on behalf of "
    "the same customer totalling GBP 25,000 or more within any rolling 30-day period "
    "must be aggregated and reviewed as a pattern regardless of individual transaction "
    "size. A pattern of cash deposits individually below GBP 10,000 that appears "
    "designed to avoid the single-transaction threshold is structuring and must be "
    "escalated for investigation. The stated indicators are deposits within 10% below "
    "the threshold, deposits on consecutive days, and deposits split across multiple "
    "branches."
)

# Every claim in the golden must be traceable to the served document.
#
# The comparison is made on whitespace-collapsed text. The served markdown hard
# wraps at roughly 72 characters, so a clause that spans a line break is not a
# raw substring of `content` even when the wording is present verbatim. Line
# break positions carry no policy meaning, so collapsing runs of whitespace
# keeps the assertion strict about wording while ignoring layout.
import re

REQUIRED_CLAUSES = [
    "GBP 10,000 or more must be recorded and reviewed before processing",
    "GBP 25,000 or more within any rolling 30-day period must be aggregated",
    "deposits within 10% below the threshold",
    "deposits on consecutive days",
    "deposits split across multiple branches",
]
aml001_flat = re.sub(r"\s+", " ", aml001_content)
missing = [c for c in REQUIRED_CLAUSES if re.sub(r"\s+", " ", c) not in aml001_flat]
if missing:
    raise RuntimeError(
        "The golden below cites clauses that are not in the document the API serves:\n  "
        + "\n  ".join(missing)
        + "\nThe comparison already ignores line breaks and repeated spaces, so this "
          "means the wording itself is absent: the policy corpus has changed. Update "
          "the golden from the current document text rather than weakening the "
          "assertion."
    )
print("\nAll golden clauses verified present in the served AML-001 text.")
print()
print("EXPECTED OUTPUT (golden)")
print(textwrap.fill(EXPECTED_OUTPUT, width=96, initial_indent="  ", subsequent_indent="  "))

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

retrieval_context = [chunk["text"] for chunk in retrieval["chunks"]]

test_case = LLMTestCase(
    input=retrieval["query"],
    actual_output=EXPECTED_OUTPUT,   # not scored by this metric; keeps the case well-formed
    expected_output=EXPECTED_OUTPUT,
    retrieval_context=retrieval_context,
)

print("USER INPUT")
print(" ", test_case.input)
print()
print("EXPECTED OUTPUT (golden)")
print(textwrap.fill(test_case.expected_output, width=96, initial_indent="  ",
                    subsequent_indent="  "))
print()
print(f"RETRIEVAL CONTEXT ({len(retrieval_context)} nodes, in retrieval rank order)")
for rank, (chunk, text) in enumerate(zip(retrieval["chunks"], retrieval_context), start=1):
    print(f"  rank {rank:>2}  {chunk['chunk_id']:<12} {text[:100].replace(chr(10), ' ')}...")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `ContextualPrecisionMetric`.

Kept at the default on purpose. The score is a weighted cumulative precision over the
whole returned list, so it is depressed by every irrelevant node the caller asked for -
and this request deliberately asks for `top_k: 10` when only one or two chunks are
relevant. A high threshold on a deliberately padded list would measure the `top_k` choice
rather than the retriever. `0.5` asks the meaningful question: *are the relevant chunks
near the top?*

In [ ]:
from deepeval.metrics import ContextualPrecisionMetric

metric = ContextualPrecisionMetric(
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **The golden is a human artefact.** Precision is measured *relative to
   `expected_output`*. A vaguely written golden makes almost any chunk look relevant and
   inflates the score. The assertion cell above defends against drift, not against a badly
   chosen reference answer.
2. **Text-level judging discards the ids.** The application exposes `chunk_id` and
   `score`, which would support an exact, judge-free ranking assertion against a list of
   expected chunk ids. This metric does not use them - if you need a deterministic
   retrieval gate, assert on `chunk_id` ordering directly and keep this metric for the
   semantic view.
3. **`top_k` moves the score.** A larger `top_k` adds irrelevant tail nodes and lowers
   precision. Comparisons across runs are only meaningful with `top_k` held fixed; it is
   pinned in the request cell for that reason.
4. **Chunk ids are stable only within a seed version.** They are `doc{id}-c{index}` and a
   reseed, re-chunk or embedding change invalidates any retrieval baseline. The seed guard
   in the scenario cell fails loudly rather than quietly comparing against a moved corpus.
5. **Scoped retrieval is a different question.** This run filters to `document_type:
   "policy"`. Case-scoped evidence retrieval ranks a different corpus and needs its own
   golden; a single precision number does not generalise across both.